# Libraries

In [1]:
# Import Packages
import os
import numpy as np
import polars as pl
import pyarrow.dataset as pads
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# import sklearn.metrics

# Establish OF Pos

In [2]:
# Read in what pos to process
of_pos_file = open("of_pos.txt", "r")
of_pos = int(of_pos_file.read())
of_pos_file.close()

# Read in Data

In [ ]:
####################################################
### Code Taken from SMT_Data_Starter
####################################################

# For the data_path argument, include the full file path to the folder that holds the data!    
def readDataSubset(table_type, data_path=):
    '''
    Takes a the set of tables from Data set and
    transforms into panda data frame for manipulation
    
    Input:
    -     table_type   = Set of tables of interest from Data set [str]
    -     data_path    = File Path which data is located in Local device, change as necessary[str]
    
    '''
    if table_type not in ['ball-positions', 'ball-events', 'player-positions', 'lineups', 'game-info']:
        print("Invalid data subset name. Please try again with a valid data subset.")
        return -1

    if table_type == 'lineups' or table_type == 'game-info':
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [4]:
####################################################
### Code Taken from SMT_Data_Starter
####################################################

# Read in the Ball Events Data
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table().to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Read in the Ball Pos Data
ball_pos_subset = readDataSubset('ball-positions')
ball_pos = ball_pos_subset.to_table().to_pandas()
ball_pos_df = pl.from_pandas(ball_pos)

# Read in the Player Pos Data for chosen OF pos
player_pos_subset = readDataSubset('player-positions')
player_pos = player_pos_subset.to_table(filter = (pads.field("player_id") == of_pos)).to_pandas()
player_pos_df = pl.from_pandas(player_pos)

# Find Timestamps for Contact and Next Event

In [34]:
# Find the timestamp for contact of all hits that either bounce off the ground or are caught by the targeted fielder
bbip_hit_df = ball_events_df \
                               .filter(
                                   (pl.col("player_id") == 10) &
                                   (pl.col("ball_eventcode") == 4) &
                                   (pl.col("player_id").shift(-1).is_in([of_pos, 255])) &
                                   (pl.col("ball_eventcode").shift(-1).is_in([2, 16]))
                                  ) \
                                .with_columns(
                                    pl.col("timestamp").alias("hit_timestamp")
                                ) \
                                .select(
                                    ["game_string", "play_per_game", "hit_timestamp"]
                                ) \
                                .drop_nulls(
                                ) \
                                .sort(
                                    ["game_string", "play_per_game"]
                                     )

# Find the timestamp after contact of all hits that either bounce off the ground or are caught by the targeted fielder to give us the next event
bbip_next_event_df = ball_events_df \
                               .filter(
                                   (pl.col("player_id").shift(1) == 10) &
                                   (pl.col("ball_eventcode").shift(1) == 4) &
                                   (pl.col("player_id").is_in([of_pos, 255])) &
                                   (pl.col("ball_eventcode").is_in([2, 16]))
                                  ) \
                                .with_columns(
                                    pl.col("timestamp").alias("next_event_timestamp")
                                ) \
                                .select(
                                      ["game_string", "play_per_game", "player_id", "ball_eventcode", "next_event_timestamp"]
                                )\
                                .drop_nulls(
                                ) \
                                .sort(
                                    ["game_string", "play_per_game"]
                                )

game_string,play_per_game,player_id,ball_eventcode,next_event_timestamp
str,f64,i64,i64,f64
"""y1_d061_VKA_PHD""",2.0,255,16,76464.0
"""y1_d061_VKA_PHD""",3.0,255,16,110064.0
"""y1_d061_VKA_PHD""",11.0,255,16,396064.0
"""y1_d061_VKA_PHD""",22.0,255,16,720514.0
"""y1_d061_VKA_PHD""",25.0,255,16,913914.0
…,…,…,…,…
"""y1_d217_VAS_ARN""",307.0,255,16,9.857836e6
"""y1_d217_VAS_ARN""",316.0,9,2,1.0070386e7
"""y1_d217_VAS_ARN""",321.0,255,16,1.0209736e7


# Calculate Hang Time

In [6]:
# Calculate hang time between the two events
hang_times_df = bbip_next_event_df \
                                .join(
                                  bbip_hit_df,
                                  how = "left",
                                  on = ["game_string", "play_per_game"]) \
                                 .with_columns(
                                     hang_time = ((pl.col("next_event_timestamp") - pl.col("hit_timestamp")) / 1000).round(2)\
                                         .over(["game_string", "play_per_game"])
                                 ) \
                                 .filter( # Filter out hits with missing data
                                     pl.col("hang_time") <= 20,
                                     pl.col("hang_time") >= 0,
                                     (pl.col("ball_eventcode").is_in([2, 16]))
                                 ) \
                                .select(["game_string", "play_per_game", "ball_eventcode", "hit_timestamp", "next_event_timestamp", "hang_time", "player_id"]
                                       )

# Calculate Hit Distance

In [24]:
# Add in ball position data to hit and next event timestamp
hit_dist_df = ball_pos_df \
                         .join(
                              hang_times_df,
                              how = "inner",
                              on = ["game_string", "play_per_game"]) \
                         .filter(
                             (pl.col("timestamp") == pl.col("hit_timestamp")) |
                             (pl.col("timestamp") == pl.col("next_event_timestamp"))
                         ) \
                         .with_columns( # calculate the change between the two timestamps
                             change_x = (pl.col("ball_position_x").shift(-1) - pl.col("ball_position_x")).over(["game_string", "play_per_game"]),
                             change_y = (pl.col("ball_position_y").shift(-1) - pl.col("ball_position_y")).over(["game_string", "play_per_game"])
                         ) \
                         .with_columns( # gives us hit distance
                             hit_distance = (pl.col("change_x") ** 2 + pl.col("change_y")** 2).sqrt().over(["game_string", "play_per_game"])
                         ) \
                         .select(
                             ["game_string", "play_per_game", "hang_time", "ball_eventcode", "hit_distance", "change_x", "change_y", "timestamp",
                              "hit_timestamp", "next_event_timestamp", "player_id"]
                         ) \
                         .filter( # filter for balls > 175 to isolate balls hit to the outfield
                             (pl.col("hit_distance") > 175) &
                             (pl.col("hit_distance").is_not_null())
                         ).sort(["game_string", "play_per_game"])

game_string,play_per_game,timestamp,ball_position_x,ball_position_y,ball_position_z,home_team,away_team,year,day,ball_eventcode,hit_timestamp,next_event_timestamp,hang_time,player_id,change_x,change_y,hit_distance
str,f64,f64,f64,f64,f64,str,str,str,str,i64,f64,f64,f64,i64,f64,f64,f64
"""y1_d061_VKA_PHD""",11.0,392164.0,0.1212141,4.33545,2.218167,"""PHD""","""VKA""","""y1""","""d061""",16,392164.0,396064.0,3.9,255,40.799386,371.14755,373.383307
"""y1_d061_VKA_PHD""",22.0,717914.0,0.0918126,0.2925132,1.549938,"""PHD""","""VKA""","""y1""","""d061""",16,717914.0,720514.0,2.6,255,-179.339113,244.076687,302.879425
"""y1_d061_VKA_PHD""",28.0,973664.0,-0.514089,3.77661,3.25038,"""PHD""","""VKA""","""y1""","""d061""",2,973664.0,979614.0,5.95,9,63.838389,186.11079,196.755092
"""y1_d061_VKA_PHD""",64.0,2.016264e6,0.687837,1.285587,1.820307,"""PHD""","""VKA""","""y1""","""d061""",16,2.016264e6,2.018764e6,2.5,255,-3.897357,199.362813,199.400904
"""y1_d061_VKA_PHD""",88.0,2.840064e6,0.2018127,4.20006,2.320671,"""PHD""","""VKA""","""y1""","""d061""",16,2.840064e6,2.841914e6,1.85,255,-110.051613,205.65804,233.252196
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""y1_d217_VAS_ARN""",266.0,8.565486e6,0.46833,1.898598,1.643781,"""ARN""","""VAS""","""y1""","""d217""",16,8.565486e6,8.567286e6,1.8,255,-90.70953,184.886802,205.940157
"""y1_d217_VAS_ARN""",290.0,9.324686e6,-0.238297,1.340433,2.208387,"""ARN""","""VAS""","""y1""","""d217""",16,9.324686e6,9.329136e6,4.45,255,107.275897,366.762567,382.129427
"""y1_d217_VAS_ARN""",307.0,9.855686e6,0.409482,-0.381171,2.882862,"""ARN""","""VAS""","""y1""","""d217""",16,9.855686e6,9.857836e6,2.15,255,-121.588482,205.439871,238.724317


# Find All Relevant Timestamps

In [19]:
# Add in relevant player pos data 
player_dist_df = player_pos_df\
                            .join(
                                hit_dist_df,
                                how = "inner",
                                on = ["game_string", "play_per_game"]
                            ) \
                            .filter( # all relevant time stamps
                                pl.col("player_id") == of_pos,
                                (pl.col("timestamp") >= pl.col("hit_timestamp")) |
                                (pl.col("timestamp") <= pl.col("next_event_timestamp"))   
                            ) \
                            .select(
                                  ["game_string", "play_per_game", "timestamp", "hit_timestamp", "next_event_timestamp",
                                  "hang_time", "field_x", "field_y", "hit_distance", "ball_eventcode", "player_id"]
                            )

game_string,play_per_game,timestamp,hit_timestamp,next_event_timestamp,hang_time,field_x,field_y,hit_distance,ball_eventcode,player_id
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""y1_d143_ADQ_ANI""",17.0,374004.0,377454.0,379754.0,2.3,146.5272,261.5535,229.644564,16,9
"""y1_d143_ADQ_ANI""",17.0,374054.0,377454.0,379754.0,2.3,146.5287,261.5133,229.644564,16,9
"""y1_d143_ADQ_ANI""",17.0,374104.0,377454.0,379754.0,2.3,146.5305,261.4725,229.644564,16,9
"""y1_d143_ADQ_ANI""",17.0,374154.0,377454.0,379754.0,2.3,146.5317,261.4317,229.644564,16,9
"""y1_d143_ADQ_ANI""",17.0,374204.0,377454.0,379754.0,2.3,146.5326,261.3903,229.644564,16,9
…,…,…,…,…,…,…,…,…,…,…
"""y1_d103_WTH_VAS""",301.0,9.870185e6,9.861385e6,9.863885e6,2.5,205.4406,256.3629,287.792813,16,9
"""y1_d103_WTH_VAS""",301.0,9.870235e6,9.861385e6,9.863885e6,2.5,205.2444,256.2651,287.792813,16,9
"""y1_d103_WTH_VAS""",301.0,9.870285e6,9.861385e6,9.863885e6,2.5,205.0491,256.1682,287.792813,16,9


# Join with Ball Pos Data

In [20]:
# Add in relevant ball pos data 
player_ball_df = player_dist_df \
                                .join(
                                    ball_pos_df,
                                    how = "inner",
                                    on = ["game_string", "play_per_game", "timestamp"]
                                ) \
                                .select(
                                      ["game_string", "play_per_game", "timestamp", "hit_timestamp", "next_event_timestamp", "ball_eventcode", "player_id",
                                      "hang_time", "field_x", "field_y", "hit_distance", "ball_position_x", "ball_position_y", "ball_position_z"]
                                )

game_string,play_per_game,timestamp,hit_timestamp,next_event_timestamp,ball_eventcode,player_id,hang_time,field_x,field_y,hit_distance,ball_position_x,ball_position_y,ball_position_z
str,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""y1_d143_ADQ_ANI""",17.0,377004.0,377454.0,379754.0,16,9,2.3,144.8307,256.6494,229.644564,-1.611609,54.177,6.31017
"""y1_d143_ADQ_ANI""",17.0,377054.0,377454.0,379754.0,16,9,2.3,144.7677,256.5711,229.644564,-1.479819,48.1371,6.16053
"""y1_d143_ADQ_ANI""",17.0,377104.0,377454.0,379754.0,16,9,2.3,144.702,256.4994,229.644564,-1.324644,42.1458,5.92881
"""y1_d143_ADQ_ANI""",17.0,377154.0,377454.0,379754.0,16,9,2.3,144.6336,256.4337,229.644564,-1.146081,36.2028,5.61504
"""y1_d143_ADQ_ANI""",17.0,377204.0,377454.0,379754.0,16,9,2.3,144.5625,256.3737,229.644564,-0.94413,30.3081,5.21919
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""y1_d103_WTH_VAS""",301.0,9.869485e6,9.861385e6,9.863885e6,16,9,2.5,208.1826,257.7291,287.792813,106.1277,137.8665,-0.4962
"""y1_d103_WTH_VAS""",301.0,9.869535e6,9.861385e6,9.863885e6,16,9,2.5,207.9849,257.6307,287.792813,104.3982,135.3654,-0.37215
"""y1_d103_WTH_VAS""",301.0,9.869585e6,9.861385e6,9.863885e6,16,9,2.5,207.7896,257.5332,287.792813,102.6687,132.8643,-0.2481


# Find Angle of Descent

In [26]:
# grab the tail of the relevant time stamps
ball_pos_last_20 = player_ball_df \
                                .sort(
                                    ["game_string", "play_per_game", "timestamp"]
                                ) \
                                .group_by(
                                    ["game_string", "play_per_game"]
                                ) \
                                .tail(20)
# calc angle of descent
descent_by_play = ball_pos_last_20 \
                              .with_columns( # calc change in x, y, z during the tail.
                                  change_x = (pl.col("ball_position_x").max() - pl.col("ball_position_x").min()).over(["game_string", "play_per_game"]),
                                  change_y = (pl.col("ball_position_y").max() - pl.col("ball_position_y").min()).over(["game_string", "play_per_game"]),
                                  change_z = (pl.col("ball_position_z").min() - pl.col("ball_position_z").max()).over(["game_string", "play_per_game"]),
                                  ) \
                              .with_columns( # ground distance covered
                                  ground_distance = (pl.col("change_x") ** 2 + pl.col("change_y") ** 2).sqrt()
                              ) \
                              .with_columns( # drop over ground distance
                                  descent_angle_deg = pl.arctan2("change_z", "ground_distance").degrees()
                              ) \
                              .select(
                                  ["game_string", "play_per_game", "hang_time", "hit_distance", "descent_angle_deg", "ball_eventcode"]
                              ) \
                              .unique( # once per play
                              )

game_string,play_per_game,hang_time,hit_distance,descent_angle_deg,ball_eventcode
str,f64,f64,f64,f64,i64
"""y1_d139_VAS_ARN""",130.0,2.35,273.836537,-13.768121,16
"""y1_d139_VAS_ARN""",130.0,2.35,273.836537,-13.768121,16
"""y1_d139_VAS_ARN""",130.0,2.35,273.836537,-13.768121,16
"""y1_d139_VAS_ARN""",130.0,2.35,273.836537,-13.768121,16
"""y1_d139_VAS_ARN""",130.0,2.35,273.836537,-13.768121,16
…,…,…,…,…,…
"""y1_d106_OER_ANI""",218.0,2.28,217.52153,-12.688195,16
"""y1_d106_OER_ANI""",218.0,2.28,217.52153,-12.688195,16
"""y1_d106_OER_ANI""",218.0,2.28,217.52153,-12.688195,16


# Find Optimal Route Distance and Attack Angle

In [31]:
# Isolate OFer position at contact
starting_pos_df = player_ball_df \
                            .filter(
                                pl.col("timestamp") == pl.col("hit_timestamp"),
                            ) \
                            .select(
                                ["game_string", "play_per_game", "timestamp", "field_x", "field_y", "ball_eventcode", "hit_timestamp", "player_id"]
                            )
# Isolate ball position at next event
ball_pos_next_df = player_ball_df \
                            .filter(
                                pl.col("timestamp") == pl.col("next_event_timestamp").over(["game_string", "play_per_game"]),
                            ) \
                            .select(
                                ["game_string", "play_per_game", "hang_time", "ball_position_x", "ball_position_y", "next_event_timestamp"]
                            )
# Join and calculate ORD and Atk Angle
ord_atk_df = starting_pos_df \
                            .join(
                                 ball_pos_next_df,
                                 how = "inner",
                                 on = ["game_string", "play_per_game"] 
                                ) \
                            .with_columns( # Change between timestamps
                                change_x = pl.col("field_x") - pl.col("ball_position_x"),
                                change_y = pl.col("field_y") - pl.col("ball_position_y"),
                            ) \
                            .with_columns( # Angle from fielder position to ball ending position
                                attack_angle_deg = (pl.arctan2(((pl.col("field_y")) - (pl.col("ball_position_y"))),
                                ((pl.col("field_x")) - (pl.col("ball_position_x")))).degrees()) \
                                .over(["game_string", "play_per_game"]),
                                # Distance formula for player distance to ball end point
                                optimal_route_distance = ((pl.col("change_x") ** 2) + (pl.col("change_y") ** 2)).sqrt() \
                                .over(["game_string", "play_per_game"])
                            ) \
                            .select(
                                ["game_string", "play_per_game", "optimal_route_distance", "attack_angle_deg"]
                            ) \
                            .sort(
                                ["game_string", "play_per_game"]
                                )

# Join ORD and Attack Angle with Descent

In [33]:
# Combine into full data frame, add catch status, and shrink to unique plays
full_df = descent_by_play \
                        .join(
                            ord_atk_df,
                            how = "left",
                            on = ["game_string", "play_per_game"]
                        ) \
                        .with_columns( # catch status
                            catch_status =  
                                pl.when(pl.col("ball_eventcode") == 2)
                                .then(1)
                                .otherwise(0)
                        ) \
                        .sort(["game_string", "play_per_game"]) \
                        .unique(
                            subset= ["game_string", "play_per_game"],
                            keep="first"
                        ) \
                        .drop_nulls(
                        )

game_string,play_per_game,hang_time,hit_distance,descent_angle_deg,ball_eventcode,optimal_route_distance,attack_angle_deg,catch_status
str,f64,f64,f64,f64,i64,f64,f64,i32
"""y1_d171_IGL_ANI""",107.0,5.2,319.39978,-57.719015,2,20.331688,-124.807652,1
"""y1_d126_MEX_ANI""",188.0,3.1,252.112726,-4.655585,16,217.349825,14.914112,0
"""y1_d169_MPC_PHD""",70.0,3.83,341.955072,-8.737053,16,82.899802,-66.0605,0
"""y1_d074.5_AIP_ARN""",186.0,4.15,213.888599,-8.743861,16,292.434064,20.82061,0
"""y1_d190_PHD_ARN""",140.0,2.71,240.32247,-4.274787,16,93.996782,107.119615,0
…,…,…,…,…,…,…,…,…
"""y1_d209_QHX_ANI""",290.0,4.3,240.732786,-12.467335,16,168.960831,7.646273,0
"""y1_d077_PHD_VAS""",179.0,2.2,198.969575,-4.349411,16,84.452491,55.673452,0
"""y1_d207_ATL_PHD""",88.0,3.0,261.426134,-8.707402,16,273.653056,8.967193,0


# Train/Test Split

In [14]:
# Set seed
np.random.seed(27) # My lucky number :)

# Convert df to numpy
np_df = full_df[["optimal_route_distance", "hang_time", "descent_angle_deg", "attack_angle_deg", "catch_status"]].to_numpy()

# Specify X and Y
x = np_df[:, [0, 1, 2, 3]]
y = np_df[:, 4]

# Split data into training and testing, 70/30
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.3
)

# Logistic Regression

In [35]:
# Logistic Regression Model
logistic_regression_model = LogisticRegression().fit(x_train, y_train)

# # Look Inside the Model
# logistic_regression_model

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

# Create Table for Metrics

In [16]:
# catch_prob_training_results = pl.DataFrame({
#     "actual" : y_train,
#     "logistic_reg_class" : logistic_regression_model.predict(x_train),
#     "logistic_reg_prob" : logistic_regression_model.predict_proba(x_train)
# })
# catch_prob_testing_results = pl.DataFrame({
#     "actual" : y_test,
#     "logistic_reg_class" : logistic_regression_model.predict(x_test),
#     "logistic_reg_prob" : logistic_regression_model.predict_proba(x_test),
# })

# # Combine into 1 data frame
# full_catch_prob_results = pl.concat(
#     [catch_prob_training_results, catch_prob_testing_results],
#     how = "vertical"
# )

# full_catch_prob_results

# Model Evaluation - Confusion Matrix

In [17]:
# # Confusion Matrix for Logistic Regression Model
# logistic_regression_cm = sklearn.metrics.confusion_matrix(full_catch_prob_results["actual"], full_catch_prob_results["logistic_reg_class"])

# logistic_regression_cm

# Model Evaluation Metrics

In [18]:
# accuracy = sklearn.metrics.accuracy_score(full_catch_prob_results["actual"], full_catch_prob_results["logistic_reg_class"])

# precision = sklearn.metrics.precision_score(full_catch_prob_results["actual"], full_catch_prob_results["logistic_reg_class"])

# recall = sklearn.metrics.recall_score(full_catch_prob_results["actual"], full_catch_prob_results["logistic_reg_class"])

# rmse = sklearn.metrics.root_mean_squared_error(full_catch_prob_results["actual"], full_catch_prob_results["logistic_reg_class"])

# print(f'Accuracy {accuracy:.2f},\nPrecision {precision:.2f},\nRecall {recall:.2f},\nRMSE {rmse:.2f}')